### 1. Setting up spark environment

In [0]:
from pyspark.sql import SparkSession
spark=SparkSession \
    .builder \
        .appName('Databricks_capstone') \
            .getOrCreate()


### 2. configuring storage account

In [0]:
storage_account="mystoacckad"
application_id="14a14259-70ba-4c26-a136-262468ab64da"
directory_id="26af9d76-35fe-404a-b312-869c37aec9c7"
container_name="fileshare"

service_credential = dbutils.secrets.get(scope="Secrete-scope-databricks1", key="app-reg-secrets1")

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
               f"org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", application_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", service_credential)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{directory_id}/oauth2/token")

### 3. Creating dataframes for all tables

In [0]:
folder_path=f"abfss://{container_name}@{storage_account}.dfs.core.windows.net/"
csv=[file_info.name for file_info in dbutils.fs.ls(folder_path) if (file_info.name).endswith('.csv')]

df_customers_dataset=spark.read.csv(folder_path+csv[0],header=True,inferSchema=True)
df_geolocation_dataset=spark.read.csv(folder_path+csv[1],header=True,inferSchema=True)
df_order_items_dataset=spark.read.csv(folder_path+csv[2],header=True,inferSchema=True)
df_order_payments_dataset=spark.read.csv(folder_path+csv[3],header=True,inferSchema=True)
df_order_reviews_dataset=spark.read.csv(folder_path+csv[4],header=True,inferSchema=True)
df_orders_dataset=spark.read.csv(folder_path+csv[5],header=True,inferSchema=True)
df_products_dataset=spark.read.csv(folder_path+csv[6],header=True,inferSchema=True)
df_sellers_dataset=spark.read.csv(folder_path+csv[7],header=True,inferSchema=True)
df_product_category_name_translation=spark.read.csv(folder_path+csv[8],header=True,inferSchema=True)

In [0]:
#Cashing frequently used dataFrames for better performance
df_orders_dataset.cache()
df_customers_dataset.cache()
df_order_items_dataset.cache()

### Optimizing joins for Data Integration

In [0]:
orders_items_joined_df=df_orders_dataset.join(df_order_items_dataset,"order_id","inner")

In [0]:
orders_items_products_joined_df=orders_items_joined_df.join(df_products_dataset,"product_id","inner")

In [0]:
from pyspark.sql.functions import broadcast
orders_items_products_sellers_joined_df=orders_items_products_joined_df.join(broadcast(df_sellers_dataset),"seller_id", "inner")

In [0]:
full_orders_df=orders_items_products_sellers_joined_df.join(df_customers_dataset,"customer_id","inner")

In [0]:
# geolocation data
from pyspark.sql.functions import col

full_orders_df=(full_orders_df.alias("f").join(broadcast(df_geolocation_dataset).alias("g"
), col("f.customer_zip_code_prefix") == col("g.geolocation_zip_code_prefix"), "left").select("f.*",col("g.geolocation_lat"),col("g.geolocation_lng"),col("g.geolocation_city"),col("g.geolocation_state")))

In [0]:
full_orders_df=full_orders_df.join(df_order_payments_dataset,"order_id","left")

In [0]:
full_orders_df=full_orders_df.join(broadcast(df_order_reviews_dataset),"order_id","left")

In [0]:
from pyspark.sql.functions import *
#total revenues per seller
total_revenue=full_orders_df.groupBy("seller_id").agg(sum(col("price")).alias("total_revenue"))


In [0]:
total_revenue.show()

In [0]:
#Total orders per customer
from pyspark.sql.functions import *
customer_order_count_df=full_orders_df.groupBy('customer_id') \
    .agg(count(col('order_id')).alias('Total_orders')) \
        .orderBy(desc('Total_orders'))
    
customer_order_count_df.show()

In [0]:
full_orders_df=full_orders_df.withColumn("review_score", col("review_score").cast("int"))

In [0]:
#avg review score per seller
avg_review_score=full_orders_df.groupBy('seller_id') \
    .agg(avg("review_score").alias('avg_review_score')) \
        .orderBy(col('avg_review_score').desc())

avg_review_score.show()

In [0]:
#top 10 most sold products
most_sold_products=full_orders_df.groupBy("product_id").agg(count("order_id").alias("total_sale")).orderBy(col("total_sale").desc()).limit(10)
most_sold_products.show()

In [0]:
# top 10 customers by spending
top_customers=full_orders_df \
    .groupBy('customer_id').agg(sum(col("price")).alias("total_spendings")) \
        .orderBy(col("total_spendings").desc()) \
            .limit(10)

In [0]:
top_customers.show()

### Window Functions & Ranking

In [0]:
#rank top selling products per seller
from pyspark.sql.window import Window

window_spec = Window.partitionBy("seller_id").orderBy(col("product_count").desc())

top_selling_products_per_seller = full_orders_df.groupBy("seller_id", "product_id") \
    .agg(count("order_id").alias("product_count")) \
    .withColumn("rank", rank().over(window_spec)).filter(col("rank")<=5)
display(top_selling_products_per_seller)


In [0]:
#dense rank for sellers based on revenue
window_spec=Window.orderBy(col("total_price").desc())

seller_rank_based_on_revenue=full_orders_df \
    .groupBy("seller_id") \
        .agg(sum("price").alias("total_price")) \
            .withColumn("dense_rank", dense_rank().over(window_spec)).orderBy(col("dense_rank"))

seller_rank_based_on_revenue.display()

### Advance Aggregation and Enrichment


In [0]:
#total revenue and avg order value per customer

customer_spending=full_orders_df.groupBy('customer_id') \
    .agg(count(col("order_id")).alias("total_orders"),round(sum(col("price")),2).alias("total_revenue"), round(avg(col("price")),2).alias("AOV")).orderBy(col("total_revenue").desc())

display(customer_spending)

In [0]:
#seller  performance metrics(Revenue, Average Review, Order Count)
performance_metrics=full_orders_df.groupBy("seller_id") \
    .agg(round(sum(col("price"))).alias("Revenue"), avg(col("review_score")).alias("average_review"), count(col("order_id")).alias("order_count"), round(stddev("price"),2).alias("price_variability")).orderBy(col("Revenue").desc())
    
display(performance_metrics)

In [0]:
#product popularity metrics

product_metrics_df=full_orders_df.groupBy("product_id") \
    .agg(count(col("order_id")).alias("total_sales"), 
         round(sum(col("price")),2).alias("total_revenue"), 
         round(stddev(col("price")),2).alias("price_volatility"),
         collect_set("seller_id").alias("sellers") ) \
             .orderBy(col("total_sales").desc())

In [0]:
display(product_metrics_df, truncate=False)

In [0]:
#Monthly revenue and order count trend
''''
order_purchase_timestamp--> month
total_orders
total_revenue
avg_order_value
min_order_value
max_order_value

'''

monthly_revenue_df=full_orders_df \
    .withColumn("mon",month(col("order_purchase_timestamp"))) \
        .groupBy("mon") \
            .agg(count(col("order_id")).alias("total_orders"), 
                 round(sum(col("price")),2).alias("total_revenue"), 
                 round(avg(col("price")),2).alias("avg_order_value"), 
                 round(min(col("price")),2).alias("min_order_value"), 
                 round(max(col("price")),2).alias("max_order_value")) 

display(monthly_revenue_df,truncate=False)



In [0]:
#customer retention analysis(first & last Order)


### Extended Enrichment

In [0]:
#order status flags
full_orders_df=full_orders_df \
    .withColumn("is_delivered",when(col("order_status")=="delivered", lit(1)).otherwise(lit(0))) \
        .withColumn("is_canceled",when(col("order_status")=="canceled",lit(1)).otherwise(lit(0)))

full_orders_df.filter(col("order_status") =="delivered").show()

In [0]:
#order revenue calculation
full_orders_df=full_orders_df \
    .withColumn("order_revenue", col("price")+col("freight_value")) \
        .orderBy(col("order_revenue").desc())
display(full_orders_df.select("order_revenue","price","freight_value"))

In [0]:
#customer segmentation based on spending
customer_spending=customer_spending \
    .withColumn("customer_segment", when(col("AOV")>=1200, "High_Value") \
        .when((col("AOV")<1200) & (col("AOV")>=500), "Medium_Value") \
            .otherwise("Low_Value"))
    

display(customer_spending)

In [0]:
#hourly order distribution
full_orders_df=full_orders_df.withColumn("hour_of_day",expr('hour(order_purchase_timestamp)'))
full_orders_df.select('order_purchase_timestamp','hour_of_day').show()

In [0]:
#Weekday vs Weekend order

full_orders_df=full_orders_df \
    .withColumn('order_day_type', 
                when(expr('dayofweek(order_purchase_timestamp) IN(1,7)'),
                     lit("Weekend")) \
                         .otherwise(lit("Weekday")))

full_orders_df.show()

In [0]:
#a new column freight category based on freight Value --> low, medium, high


In [0]:
#order volume by customer state